# FedAvg: Federated Averaging

Federated fraud detection on SAML-D using **FedAvg** (Federated Averaging, McMahan et al. 2017).

FedAvg is the foundational federated learning algorithm: each client trains locally for several
epochs, then the server aggregates updates by computing a **weighted average** of client weights,
with weights proportional to the number of local samples.

**Aggregation:** \( w_{t+1} = \sum_{k} \frac{n_k}{n} w_k^t \)

**FL strategy:** FedAvg (MLP)
**Tree strategies:** LightGBM · XGBoost (local ensemble via prediction averaging)
**Dataset:** SAML-D (Synthetic AML)
**Partition:** Dirichlet alpha sweep (default: [0.1])
**Weights saved to:** `weights_fedavg/`

In [ ]:
!pip install lightgbm xgboost imbalanced-learn joblib -q

## Imports

In [ ]:
import os, gc, time, warnings, random, json
from collections import defaultdict

import numpy as np
import pandas as pd

from imblearn.over_sampling import SMOTE
from joblib import Parallel, delayed

from sklearn.neural_network import MLPClassifier
from sklearn.preprocessing import LabelEncoder
from sklearn.metrics import (
    f1_score, roc_auc_score, precision_score, recall_score,
    average_precision_score, matthews_corrcoef,
    balanced_accuracy_score, confusion_matrix, fbeta_score
)

import xgboost as xgb
import lightgbm as lgb

import matplotlib
matplotlib.use('Agg')
import matplotlib.pyplot as plt

# cuML auto-detect: GPU if available, else CPU
try:
    from cuml.preprocessing import StandardScaler
    USE_GPU = True
    print("cuML available — GPU mode")
except ImportError:
    from sklearn.preprocessing import StandardScaler
    USE_GPU = False
    print("cuML not available — CPU mode")

warnings.filterwarnings('ignore')
random.seed(42)
np.random.seed(42)

print("All imports OK")

## Config

### MLP Config

In [ ]:
# ── MLP Config ───────────────────────────────────────────────────────────
MLP_HIDDEN   = (128, 64, 32)
MLP_LR       = 0.001
MLP_MAX_ITER = 5      # local training epochs per round
MLP_CAP      = 100_000

### LightGBM Config

In [ ]:
# ── LightGBM Config ──────────────────────────────────────────────────────
LGB_N_ESTIMATORS = 200
LGB_MAX_DEPTH    = 6
LGB_LR           = 0.05
LGB_SUBSAMPLE    = 0.8
LGB_COLSAMPLE    = 0.8
TREE_CAP         = 200_000

### XGBoost Config

In [ ]:
# ── XGBoost Config ───────────────────────────────────────────────────────
XGB_N_ESTIMATORS = 200
XGB_MAX_DEPTH    = 6
XGB_LR           = 0.05
XGB_TREE_METHOD  = 'gpu_hist' if USE_GPU else 'hist'

### FL & Global Config

In [ ]:
# ── FL / Global Config ───────────────────────────────────────────────────
T0  = time.time()
OUT = '/kaggle/working' if os.path.exists('/kaggle') else '.'
WEIGHTS_DIR = 'weights_fedavg'
os.makedirs(WEIGHTS_DIR, exist_ok=True)

SEED       = 42
random.seed(SEED)
np.random.seed(SEED)

def elapsed(): return f"{(time.time()-T0)/60:.1f}m"
def flush():   gc.collect()

N_BANKS    = 4
ALPHAS     = [0.1]   # add more Dirichlet alpha values here
CHUNK_SIZE = 50_000  # rows per chunk for CSV loading

FL_ROUNDS    = 20
LOCAL_EPOCHS = 5
LR           = MLP_LR
TEST_FRAC      = 0.15
VAL_FRAC       = 0.15
MIN_FRAUD      = 5
THRESH_DEFAULT = 0.5
TUNE_THRESHOLD = True
THRESH_GRID    = np.arange(0.05, 0.95, 0.05)
AP_KS          = (50, 100, 200)

HIGH_RISK_COUNTRIES = {'mexico', 'turkey', 'morocco', 'uae', 'iran',
                        'nigeria', 'russia', 'venezuela', 'north korea'}

print(f"Config OK | WEIGHTS_DIR={WEIGHTS_DIR} | SEED={SEED}")

## Data Loading & Preprocessing (SAML-D)

In [ ]:
def find_saml():
    candidates = [
        '/kaggle/input/datasets/berkanoztas/synthetic-transaction-monitoring-dataset-aml/SAML-D.csv',
    ]
    for p in candidates:
        if os.path.exists(p): return p
    for root, _, files in os.walk('.'):
        for f in files:
            if f.upper().startswith('SAML') and f.endswith('.csv'):
                return os.path.join(root, f)
    if os.path.exists('/kaggle'):
        for root, _, files in os.walk('/kaggle/input'):
            for f in files:
                if f.upper().startswith('SAML') and f.endswith('.csv'):
                    return os.path.join(root, f)
    return None


def load_saml():
    path = find_saml()
    if not path:
        raise FileNotFoundError("SAML-D.csv not found")
    print(f"Loading SAML-D (chunked): {path}")
    chunks = pd.read_csv(path, chunksize=CHUNK_SIZE, low_memory=False)
    df = pd.concat(chunks, ignore_index=True)
    df.columns = [c.strip() for c in df.columns]
    print(f"  {len(df):,} rows | fraud: {df['Is_laundering'].sum():,} "
          f"({df['Is_laundering'].mean()*100:.3f}%)")
    return df


def preprocess_saml(df):
    y   = df['Is_laundering'].astype(int).values
    typ = df['Laundering_type'].astype(str).fillna('Unknown').values
    src = np.full(len(df), 'saml', dtype=object)

    t_raw = pd.to_numeric(df['Time'], errors='coerce').fillna(0).values
    t_col = np.arange(len(df), dtype=np.int64) if np.ptp(t_raw) < 1e-6 else t_raw.astype(np.float64)

    amt       = df['Amount'].fillna(0).abs()
    amt_log   = np.log1p(amt).values
    amt_round = (amt % 1000 == 0).astype(int).values
    amt_thresh= (amt < 10000).astype(int).values

    t      = pd.to_numeric(df['Time'], errors='coerce').fillna(0)
    hr_sin = np.sin(2 * np.pi * t / 86400).values
    hr_cos = np.cos(2 * np.pi * t / 86400).values

    le_p   = LabelEncoder()
    pt_enc = le_p.fit_transform(df['Payment_type'].astype(str).fillna('Unknown'))

    sl_risk = df['Sender_bank_location'].astype(str).str.lower().apply(
        lambda x: int(any(h in x for h in HIGH_RISK_COUNTRIES))).values
    rl_risk = df['Receiver_bank_location'].astype(str).str.lower().apply(
        lambda x: int(any(h in x for h in HIGH_RISK_COUNTRIES))).values
    cross_border = (df['Sender_bank_location'].astype(str) !=
                    df['Receiver_bank_location'].astype(str)).astype(int).values
    curr_mm = (df['Payment_currency'].astype(str) !=
               df['Received_currency'].astype(str)).astype(int).values

    X = np.stack([
        amt_log, amt_round, amt_thresh,
        hr_sin, hr_cos, pt_enc.astype(float),
        sl_risk, rl_risk, cross_border, curr_mm,
    ], axis=1).astype(np.float32)

    print(f"  SAML-D features: {X.shape}")
    return X, y, typ, src, t_col

## Dirichlet Partition + Temporal Split

In [ ]:
def _allocate_fraud(n_fraud, n_banks, alpha):
    props = np.random.dirichlet([alpha] * n_banks)
    fcnts = (props * n_fraud).astype(int)
    diff  = n_fraud - fcnts.sum()
    if diff > 0:
        for _ in range(diff):  fcnts[np.argmin(fcnts)] += 1
    elif diff < 0:
        for _ in range(-diff): fcnts[np.argmax(fcnts)] -= 1
    assert fcnts.sum() == n_fraud
    return fcnts


def partition_dataset(X, y, typ, t_col, n_banks, alpha):
    np.random.seed(SEED)
    fraud_idx = np.where(y == 1)[0]; legit_idx = np.where(y == 0)[0]
    np.random.shuffle(fraud_idx);    np.random.shuffle(legit_idx)
    fcnts = _allocate_fraud(len(fraud_idx), n_banks, alpha)
    lper  = len(legit_idx) // n_banks
    banks = []; fp = lp = 0
    for i in range(n_banks):
        ln  = lper if i < n_banks - 1 else len(legit_idx) - lp
        idx = np.concatenate([fraud_idx[fp:fp+fcnts[i]], legit_idx[lp:lp+ln]])
        banks.append(_split(i, X[idx], y[idx], typ[idx], t_col[idx], f'SAML-Bank{i}'))
        fp += fcnts[i]; lp += ln
    print(f"Partition [SAML] {n_banks} banks | alpha={alpha}:")
    for b in banks:
        print(f"  {b['name']:20s}: {b['n_total']:>8,} txns | "
              f"{b['n_fraud']:>5,} fraud ({b['fraud_frac']*100:.3f}%) | "
              f"train={b['n_train_fraud']} val={b['n_val_fraud']} test={b['n_test_fraud']} fraud")
    return banks


def _split(bid, X, y, typ, t, name):
    order = np.argsort(t, kind='stable')
    X, y, typ = X[order], y[order], typ[order]
    n       = len(X)
    n_train = max(int(n * (1 - TEST_FRAC - VAL_FRAC)), 1)
    n_val   = max(int(n * VAL_FRAC), 1)
    if n_train + n_val >= n: n_train = max(n - 2, 1); n_val = 1
    Xtr, ytr, ttr = X[:n_train],              y[:n_train],              typ[:n_train]
    Xvl, yvl, tvl = X[n_train:n_train+n_val], y[n_train:n_train+n_val], typ[n_train:n_train+n_val]
    Xte, yte, tte = X[n_train+n_val:],        y[n_train+n_val:],        typ[n_train+n_val:]
    sc  = StandardScaler()
    Xtr = sc.fit_transform(Xtr).astype(np.float32)
    Xvl = sc.transform(Xvl).astype(np.float32)
    Xte = sc.transform(Xte).astype(np.float32)
    return dict(
        id=bid, name=name, source='saml',
        X_train=Xtr, y_train=ytr, typ_train=ttr,
        X_val=Xvl,   y_val=yvl,   typ_val=tvl,
        X_test=Xte,  y_test=yte,  typ_test=tte,
        n_fraud=int(y.sum()), n_total=len(y), fraud_frac=float(y.mean()),
        n_train_fraud=int(ytr.sum()), n_val_fraud=int(yvl.sum()),
        n_test_fraud=int(yte.sum()),
    )


def safe_smote(X, y):
    if y.sum() < 5 or len(X) < 20: return X, y
    try:
        k = min(5, int(y.sum()) - 1)
        Xs, ys = SMOTE(random_state=SEED, k_neighbors=k).fit_resample(X, y)
        return Xs.astype(np.float32), ys.astype(np.int64)
    except Exception:
        return X, y


def get_mlp_data(X, y, cap=MLP_CAP):
    if len(X) > cap:
        fi = np.where(y == 1)[0]; li = np.where(y == 0)[0]
        nf = min(len(fi), cap // 10); nl = min(len(li), cap - nf)
        np.random.shuffle(fi); np.random.shuffle(li)
        idx = np.concatenate([fi[:nf], li[:nl]]); np.random.shuffle(idx)
        X, y = X[idx], y[idx]
    return safe_smote(X, y)

## MLP Helpers

In [ ]:
def make_mlp():
    return MLPClassifier(
        hidden_layer_sizes=MLP_HIDDEN, activation='relu',
        solver='adam', learning_rate_init=MLP_LR,
        max_iter=MLP_MAX_ITER, random_state=SEED,
        warm_start=False, tol=1e-4)


def get_weights(m):
    return [c.copy() for c in m.coefs_] + [i.copy() for i in m.intercepts_]


def set_weights(m, w):
    n = len(m.coefs_)
    m.coefs_      = [x.copy() for x in w[:n]]
    m.intercepts_ = [x.copy() for x in w[n:]]


def init_mlp(X, y):
    if 0 not in y: X, y = np.vstack([X, X[0:1]]), np.append(y, 0)
    if 1 not in y: X, y = np.vstack([X, X[0:1]]), np.append(y, 1)
    m = make_mlp(); m.fit(X, y); return m


def clone_mlp(template, w):
    m    = make_mlp()
    tiny = np.zeros((2, template.coefs_[0].shape[0]), dtype=np.float32)
    try: m.fit(tiny, np.array([0, 1]))
    except: pass
    set_weights(m, w); return m


def mlp_proba(m, X):
    try:    return m.predict_proba(X)[:, 1]
    except: return np.full(len(X), 0.5)


def save_weights(weights, path):
    np.savez(path, *weights)


def load_weights(path):
    d = np.load(path + '.npz')
    return [d[k] for k in sorted(d.files)]


print("MLP helpers ready")

## Client Training Functions

In [ ]:
def train_mlp_client(config, X_train, y_train, X_val, y_val,
                     global_weights=None):
    X_tr, y_tr = get_mlp_data(X_train, y_train)
    if len(np.unique(y_tr)) < 2:
        return None, None, None
    init_X = np.vstack([X_tr[:10], X_tr[:10]])
    init_y = np.array([0, 1] + [int(y_tr[0])] * (len(init_X) - 2))
    m = init_mlp(init_X, init_y)
    if global_weights is not None:
        set_weights(m, global_weights)
    m.fit(X_tr, y_tr)
    lw = get_weights(m)
    try:
        probs = mlp_proba(m, X_val)
        eps = 1e-7
        val_loss = -float(np.mean(
            y_val * np.log(probs + eps) + (1 - y_val) * np.log(1 - probs + eps)
        ))
    except Exception:
        val_loss = float('inf')
    return m, lw, val_loss


def train_lgb_client(config, X_train, y_train, X_val, y_val):
    n0 = (y_train == 0).sum(); n1 = max((y_train == 1).sum(), 1)
    X_tr, y_tr = safe_smote(X_train, y_train)
    m = lgb.LGBMClassifier(
        n_estimators=LGB_N_ESTIMATORS, max_depth=LGB_MAX_DEPTH,
        learning_rate=LGB_LR, scale_pos_weight=float(n0 / n1),
        subsample=LGB_SUBSAMPLE, colsample_bytree=LGB_COLSAMPLE,
        device='gpu' if USE_GPU else 'cpu',
        random_state=SEED, verbose=-1)
    m.fit(X_tr, y_tr)
    try:
        probs = m.predict_proba(X_val)[:, 1]
        eps = 1e-7
        val_loss = -float(np.mean(
            y_val * np.log(probs + eps) + (1 - y_val) * np.log(1 - probs + eps)
        ))
    except Exception:
        val_loss = float('inf')
    return m, None, val_loss


def train_xgb_client(config, X_train, y_train, X_val, y_val):
    n0 = (y_train == 0).sum(); n1 = max((y_train == 1).sum(), 1)
    X_tr, y_tr = safe_smote(X_train, y_train)
    m = xgb.XGBClassifier(
        n_estimators=XGB_N_ESTIMATORS, max_depth=XGB_MAX_DEPTH,
        learning_rate=XGB_LR, scale_pos_weight=float(n0 / n1),
        subsample=0.8, colsample_bytree=0.8,
        tree_method=XGB_TREE_METHOD, eval_metric='aucpr',
        use_label_encoder=False, random_state=SEED, verbosity=0)
    m.fit(X_tr, y_tr)
    try:
        probs = m.predict_proba(X_val)[:, 1]
        eps = 1e-7
        val_loss = -float(np.mean(
            y_val * np.log(probs + eps) + (1 - y_val) * np.log(1 - probs + eps)
        ))
    except Exception:
        val_loss = float('inf')
    return m, None, val_loss


print("Client training functions ready")

## FedAvg Aggregation

In [ ]:
def fedavg_aggregate(global_weights, local_weights_list, sizes):
    """Weighted average of client weights (FedAvg)."""
    if not local_weights_list:
        return global_weights
    total = sum(sizes)
    n = len(local_weights_list[0])
    return [
        sum((s / total) * lw[i] for lw, s in zip(local_weights_list, sizes))
        for i in range(n)
    ]


def fedavg_aggregate_probs(prob_list, sizes):
    total = sum(sizes)
    return sum((s / total) * p for p, s in zip(prob_list, sizes))


print("FedAvg aggregation ready")

## FL Round Functions

In [ ]:
def _fedavg_train_client_mlp(idx, client_data, val_splits, global_weights, config):
    X_c, y_c = client_data[idx]
    X_vl, y_vl = val_splits[idx]
    if len(np.unique(y_c)) < 2:
        return None
    m, _, _ = train_mlp_client(config, X_c, y_c, X_vl, y_vl,
                                global_weights=global_weights)
    if m is None:
        return None
    return (get_weights(m), len(X_c))


def run_fedavg_round_mlp(client_data, val_splits, global_weights, global_model, config):
    results = Parallel(n_jobs=-1, prefer='threads')(
        delayed(_fedavg_train_client_mlp)(
            i, client_data, val_splits, global_weights, config
        )
        for i in range(len(client_data))
    )
    results = [r for r in results if r is not None]
    if not results:
        return global_weights
    return fedavg_aggregate(global_weights, [r[0] for r in results], [r[1] for r in results])


def run_fedavg_round_tree(client_data, val_splits, model_type, config):
    models = []
    for idx in range(len(client_data)):
        X_c, y_c = client_data[idx]
        X_vl, y_vl = val_splits[idx]
        if len(np.unique(y_c)) < 2:
            continue
        if model_type == 'lightgbm':
            m, _, _ = train_lgb_client(config, X_c, y_c, X_vl, y_vl)
        else:
            m, _, _ = train_xgb_client(config, X_c, y_c, X_vl, y_vl)
        models.append((m, len(X_c)))
    return models


print("FL round functions ready")

## Threshold Tuning & Metrics

In [ ]:
def tune_threshold(y_val, probs_val, default=THRESH_DEFAULT):
    if not TUNE_THRESHOLD or y_val.sum() == 0 or y_val.sum() == len(y_val):
        return default
    best_t, best_f2 = default, -1
    for t in THRESH_GRID:
        preds = (probs_val >= t).astype(int)
        if preds.sum() == 0: continue
        f2 = fbeta_score(y_val, preds, beta=2, zero_division=0)
        if f2 > best_f2: best_f2, best_t = f2, float(t)
    return best_t


TYP_W = {
    'Cycle': 2.5, 'Bipartite': 2.5, 'Stacked Bipartite': 2.5,
    'Structuring': 2.5, 'Smurfing': 2.0, 'Scatter-Gather': 2.0,
    'Gather-Scatter': 2.0, 'Fan_Out': 2.0, 'Fan_In': 2.0,
    'Layered_Fan_Out': 2.0, 'Layered_Fan_In': 2.0,
    'Deposit-Send': 2.0, 'Over-Invoicing': 2.0,
}


def compute_metrics(y_true, probs, typ=None, thresh=THRESH_DEFAULT):
    preds = (probs >= thresh).astype(int)
    n_pos = int(y_true.sum())
    tn = int(((y_true == 0) & (preds == 0)).sum())
    fp = int(((y_true == 0) & (preds == 1)).sum())
    specificity = tn / max(tn + fp, 1); fpr = fp / max(tn + fp, 1)

    if n_pos == 0:
        apk = {f'ap_at_{k}': 0. for k in AP_KS}
        return {'f1': 0., 'precision': 0., 'recall': 0., 'auprc': 0.,
                'mcc': 0., 'f2': 0., **apk,
                'typ_coverage': 0., 'typ_wf1': 0.,
                'specificity': float(specificity), 'fpr': float(fpr),
                'false_positives': fp, 'n_test_fraud': 0, 'threshold': thresh}

    f1   = f1_score(y_true, preds, zero_division=0)
    prec = precision_score(y_true, preds, zero_division=0)
    rec  = recall_score(y_true, preds, zero_division=0)
    try:    auprc = average_precision_score(y_true, probs)
    except: auprc = float(y_true.mean())
    mcc  = matthews_corrcoef(y_true, preds) if preds.sum() > 0 else 0.
    b2 = 4; d = b2 * prec + rec
    f2 = (1 + b2) * prec * rec / d if d > 0 else 0.
    sidx = np.argsort(probs)[::-1]
    apk  = {f'ap_at_{k}': float(y_true[sidx[:min(k, len(y_true))]].sum() /
                                  max(min(k, len(y_true)), 1)) for k in AP_KS}
    typ_cov = 0.; typ_wf1 = 0.
    if typ is not None:
        laund = [t for t in np.unique(typ) if t not in ('Unknown', 'Normal_Cash_Deposits')]
        if laund:
            typ_cov = sum(
                1 for t in laund
                if ((typ == t) & (y_true == 1) & (preds == 1)).sum() > 0
            ) / len(laund)
        ws = wt = 0.
        for t in np.unique(typ):
            if t in ('Unknown', 'Normal_Cash_Deposits'): continue
            mask = typ == t
            if mask.sum() < 2 or y_true[mask].sum() == 0: continue
            f = f1_score(y_true[mask], preds[mask], zero_division=0)
            w = TYP_W.get(t, 1.5); ws += w * f; wt += w
        typ_wf1 = ws / max(wt, 1e-8)

    return {'f1': f1, 'precision': prec, 'recall': rec, 'auprc': auprc,
            'mcc': mcc, 'f2': f2, **apk,
            'typ_coverage': typ_cov, 'typ_wf1': typ_wf1,
            'specificity': float(specificity), 'fpr': float(fpr),
            'false_positives': fp, 'n_test_fraud': n_pos, 'threshold': thresh}


def agg(ml):
    if not ml: return {}
    return {k: round(float(np.mean([m[k] for m in ml])), 4) for k in ml[0]}


def fairness(f1s, lf=None):
    r = dict(client_equity=round(float(np.std(f1s)), 4),
             worst_bank_f1=round(float(min(f1s)), 4),
             best_bank_f1=round(float(max(f1s)), 4))
    if lf and len(lf) == len(f1s):
        r['collab_gain'] = round(float(np.mean([a - b for a, b in zip(f1s, lf)])), 4)
    return r


print("Metrics ready")

## FedAvg Training Sweep

In [ ]:
def run_fedavg_fl(banks, config):
    """Full FedAvg training over FL_ROUNDS rounds."""
    print(f"  [FedAvg] {FL_ROUNDS} rounds", end='  ', flush=True)

    init_X = np.vstack([b['X_train'][:50] for b in banks])
    init_y = np.concatenate([b['y_train'][:50] for b in banks])
    global_model = init_mlp(init_X, init_y)
    global_weights = get_weights(global_model)

    client_data = [(b['X_train'], b['y_train']) for b in banks]
    val_splits  = [(b['X_val'],   b['y_val'])   for b in banks]

    hist = []
    for rnd in range(1, FL_ROUNDS + 1):
        global_weights = run_fedavg_round_mlp(
            client_data, val_splits, global_weights, global_model, config
        )
        if rnd % 5 == 0 or rnd == FL_ROUNDS:
            print(f"{rnd}", end=' ', flush=True)
            gm = clone_mlp(global_model, global_weights)
            for b in banks:
                yt = b['y_test']
                probs = mlp_proba(gm, b['X_test'])
                preds = (probs >= THRESH_DEFAULT).astype(int)
                f1_val    = f1_score(yt, preds, zero_division=0) if yt.sum() > 0 else 0.
                auprc_val = average_precision_score(yt, probs)   if yt.sum() > 0 else 0.
                hist.append(dict(
                    round=rnd, bank_id=b['id'], bank_name=b['name'],
                    strategy='fedavg', source='saml',
                    f1=f1_val, auprc=auprc_val,
                    n_test_fraud=int(yt.sum())
                ))
        flush()

    print()
    return clone_mlp(global_model, global_weights), global_weights, hist


def run_tree_ensemble(banks, model_type, config):
    print(f"  [FedAvg-{model_type}] training per-client trees...", end=' ')
    client_data = [(b['X_train'], b['y_train']) for b in banks]
    val_splits  = [(b['X_val'],   b['y_val'])   for b in banks]
    models = run_fedavg_round_tree(client_data, val_splits, model_type, config)
    print(f"{len(models)} models trained")
    return models


def predict_tree_ensemble(models, X):
    if not models:
        return np.full(len(X), 0.5)
    return fedavg_aggregate_probs(
        [m.predict_proba(X)[:, 1] for m, _ in models],
        [s for _, s in models]
    )


print("FedAvg training functions ready")

## Evaluation

In [ ]:
def evaluate_fedavg(banks, mlp_model, lgb_models, xgb_models):
    rows = []
    total_fraud = sum(int(b['y_test'].sum()) for b in banks)

    # ── fedavg_mlp ──
    bm = []
    for bidx, b in enumerate(banks):
        yt  = b['y_test']
        tte = b.get('typ_test')
        pv = mlp_proba(mlp_model, b['X_val'])
        pt = mlp_proba(mlp_model, b['X_test'])
        t_tune = tune_threshold(b['y_val'], pv)
        bm.append(compute_metrics(yt, pt, tte, thresh=t_tune))
    a  = agg(bm)
    fa = fairness([m['f1'] for m in bm])
    rows.append({
        'strategy': 'fedavg_mlp', 'model_type': 'tree',
        **a, **fa,
        'n_eval_banks': len(banks),
        'n_banks_with_fraud': sum(1 for b in banks if b['y_test'].sum() > 0),
        'total_test_fraud': total_fraud,
    })

    # ── fedavg_lgb ──
    bm = []
    for bidx, b in enumerate(banks):
        yt  = b['y_test']
        tte = b.get('typ_test')
        pv = predict_tree_ensemble(lgb_models, b['X_val'])
        pt = predict_tree_ensemble(lgb_models, b['X_test'])
        t_tune = tune_threshold(b['y_val'], pv)
        bm.append(compute_metrics(yt, pt, tte, thresh=t_tune))
    a  = agg(bm)
    fa = fairness([m['f1'] for m in bm])
    rows.append({
        'strategy': 'fedavg_lgb', 'model_type': 'tree',
        **a, **fa,
        'n_eval_banks': len(banks),
        'n_banks_with_fraud': sum(1 for b in banks if b['y_test'].sum() > 0),
        'total_test_fraud': total_fraud,
    })

    # ── fedavg_xgb ──
    bm = []
    for bidx, b in enumerate(banks):
        yt  = b['y_test']
        tte = b.get('typ_test')
        pv = predict_tree_ensemble(xgb_models, b['X_val'])
        pt = predict_tree_ensemble(xgb_models, b['X_test'])
        t_tune = tune_threshold(b['y_val'], pv)
        bm.append(compute_metrics(yt, pt, tte, thresh=t_tune))
    a  = agg(bm)
    fa = fairness([m['f1'] for m in bm])
    rows.append({
        'strategy': 'fedavg_xgb', 'model_type': 'tree',
        **a, **fa,
        'n_eval_banks': len(banks),
        'n_banks_with_fraud': sum(1 for b in banks if b['y_test'].sum() > 0),
        'total_test_fraud': total_fraud,
    })

    return rows


print("Evaluation function ready")

## Plotting

In [ ]:
COLORS = {'fedavg_mlp': '#1B4F72', 'fedavg_lgb': '#2E86C1', 'fedavg_xgb': '#85C1E9'}


def plot_fedavg_results(df_bm, fl_hist, tag):
    strats = sorted(df_bm['strategy'].unique())
    fig, axes = plt.subplots(1, 3, figsize=(20, 6))
    fig.suptitle(
        f'FedAvg — {tag} — SAML-D · Temporal Split · Dirichlet\n'
        'Federated Averaging (MLP) + LightGBM ensemble + XGBoost ensemble',
        fontsize=10, fontweight='bold'
    )

    ax = axes[0]; x = np.arange(len(strats)); w = 0.22
    for mi, (m, lbl, col) in enumerate([
        ('auprc', 'AUPRC', '#2E4057'),
        ('f2',    'F2',    '#048A81'),
        ('mcc',   'MCC',   '#54C6EB')
    ]):
        vals = [float(df_bm[df_bm['strategy'] == s][m].mean()) for s in strats]
        ax.bar(x + (mi - 1) * w, vals, w, label=lbl, color=col, alpha=0.85)
    ax.set_xticks(x)
    ax.set_xticklabels(strats, fontsize=8, rotation=30, ha='right')
    ax.set_ylim(0, 1); ax.legend(fontsize=8, frameon=False)
    ax.set_title('Primary Metrics (avg all banks)', fontsize=9, fontweight='bold')
    ax.spines[['top', 'right']].set_visible(False); ax.grid(axis='y', alpha=0.25)

    ax = axes[1]
    for si, s in enumerate(strats):
        col = COLORS.get(s, '#999')
        ap  = float(df_bm[df_bm['strategy'] == s]['auprc'].mean())
        rc  = float(df_bm[df_bm['strategy'] == s]['recall'].mean())
        ax.bar(si - 0.18, ap, 0.35, color=col, alpha=0.85)
        ax.bar(si + 0.18, rc, 0.35, color=col, alpha=0.4,
               edgecolor=col, linewidth=1.5)
    ax.set_xticks(range(len(strats)))
    ax.set_xticklabels(strats, fontsize=8, rotation=30, ha='right')
    ax.set_ylim(0, 1)
    ax.set_title('AUPRC (solid) vs Recall (hollow)', fontsize=9, fontweight='bold')
    ax.spines[['top', 'right']].set_visible(False); ax.grid(axis='y', alpha=0.25)

    ax = axes[2]
    dfh = pd.DataFrame(fl_hist)
    if not dfh.empty:
        g = dfh.groupby('round')['auprc'].mean().reset_index()
        ax.plot(g['round'], g['auprc'], label='FedAvg MLP',
                color=COLORS.get('fedavg_mlp', '#333'), lw=2, marker='o', ms=4)
    ax.set_xlabel('FL Round'); ax.set_ylim(0, 1)
    ax.set_title('FedAvg Learning Curve (avg AUPRC)', fontsize=9, fontweight='bold')
    ax.legend(fontsize=8, frameon=False)
    ax.spines[['top', 'right']].set_visible(False); ax.grid(alpha=0.25)

    plt.tight_layout()
    p = f'{WEIGHTS_DIR}/{tag}_fedavg_results.png'
    plt.savefig(p, dpi=150, bbox_inches='tight'); plt.close()
    print(f"Saved: {p}")


def print_fedavg_summary(df_bm, label):
    print(f"\n{'='*70}")
    print(f"FedAvg — {label} — SAML-D · Temporal Split · Dirichlet")
    print('=' * 70)
    cols = ['strategy', 'auprc', 'mcc', 'f2', 'f1', 'recall',
            'specificity', 'fpr', 'ap_at_100', 'typ_wf1',
            'client_equity', 'worst_bank_f1',
            'threshold', 'n_eval_banks', 'n_banks_with_fraud', 'total_test_fraud']
    avail = [c for c in cols if c in df_bm.columns]
    print(df_bm[avail].sort_values('auprc', ascending=False).to_string(index=False))
    best = df_bm.loc[df_bm['auprc'].idxmax()]
    print(f"\nBEST: {best['strategy']}  AUPRC={best['auprc']:.4f}  "
          f"MCC={best['mcc']:.4f}  F2={best['f2']:.4f}")


print("Plotting functions ready")

## Main: FedAvg Alpha Sweep

In [ ]:
# ================================================================
# MAIN: FedAvg sweep — SAML-D × Dirichlet alphas
# ================================================================
print("Loading SAML-D dataset...")
df_raw = load_saml()
X, y, typ, src_col, t_col = preprocess_saml(df_raw)
del df_raw; flush()
print(f"Preprocessing done | {elapsed()}")

config = {}
all_combined = []

for alpha in ALPHAS:
    tag = f"saml_alpha{alpha}"
    print(f"\n{'='*55}")
    print(f"  FedAvg | SAML-D | alpha={alpha} | {N_BANKS} banks")
    print(f"{'='*55}")

    banks = partition_dataset(X, y, typ, t_col, N_BANKS, alpha)

    print(f"\nFedAvg MLP training:")
    mlp_model, global_weights, fl_hist = run_fedavg_fl(banks, config)

    w_path = os.path.join(WEIGHTS_DIR, f'{tag}_fedavg_mlp')
    save_weights(global_weights, w_path)
    print(f"  Weights saved: {w_path}.npz")

    for h in fl_hist:
        h['alpha'] = alpha

    print(f"\nFedAvg LightGBM ensemble:")
    lgb_models = run_tree_ensemble(banks, 'lightgbm', config)

    print(f"FedAvg XGBoost ensemble:")
    xgb_models = run_tree_ensemble(banks, 'xgboost', config)

    print("\nEvaluating...")
    bm_rows = evaluate_fedavg(banks, mlp_model, lgb_models, xgb_models)
    for r in bm_rows:
        r['alpha'] = alpha; r['dataset'] = 'SAML'

    df_bm = pd.DataFrame(bm_rows)
    df_bm.to_csv(f'{WEIGHTS_DIR}/{tag}_fedavg_benchmark.csv', index=False)
    pd.DataFrame(fl_hist).to_csv(f'{WEIGHTS_DIR}/{tag}_fedavg_fl_history.csv', index=False)

    print_fedavg_summary(df_bm, f"alpha={alpha}")
    plot_fedavg_results(df_bm, fl_hist, tag)

    all_combined.append(df_bm)
    del banks, mlp_model, lgb_models, xgb_models, fl_hist; flush()
    print(f"  Done | {elapsed()}")

combined = pd.concat(all_combined, ignore_index=True)
combined.to_csv(f'{WEIGHTS_DIR}/fedavg_saml_combined.csv', index=False)

print(f"\n{'='*55}")
print("FEDAVG SWEEP COMPLETE")
print(f"{'='*55}")
print(f"Total runtime: {elapsed()}")
print(f"\nOutputs in '{WEIGHTS_DIR}':")
for f in sorted(os.listdir(WEIGHTS_DIR)):
    print(f"  {f}")

## Consolidated Results Summary

In [ ]:
combined = pd.read_csv(f'{WEIGHTS_DIR}/fedavg_saml_combined.csv')

print("FedAvg — SAML-D · All Alphas · AUPRC Pivot")
piv = combined.pivot_table(index='alpha', columns='strategy',
                            values='auprc', aggfunc='mean').round(4)
print(piv.to_string())

print("\nFedAvg — SAML-D · All Alphas · MCC Pivot")
piv_mcc = combined.pivot_table(index='alpha', columns='strategy',
                                values='mcc', aggfunc='mean').round(4)
print(piv_mcc.to_string())

print("\nFedAvg — SAML-D · All Alphas · F2 Pivot")
piv_f2 = combined.pivot_table(index='alpha', columns='strategy',
                               values='f2', aggfunc='mean').round(4)
print(piv_f2.to_string())

print("\n── Top 3 Combinations by AUPRC ─────────────────────")
top3 = combined.nlargest(3, 'auprc')[['strategy', 'alpha', 'auprc', 'mcc', 'f2']]
print(top3.to_string(index=False))
print("\nFinal weights are saved in:", WEIGHTS_DIR)